# Airspace Conflict Detection

HyPlan's `airspace` module checks whether flight lines pass through
restricted, prohibited, or controlled airspace using data from the
[OpenAIP](https://www.openaip.net/) API. It provides:

1. **`Airspace`** — a dataclass representing an airspace volume (lateral polygon + vertical limits in feet)
2. **`OpenAIPClient`** — fetches real airspace data with local caching
3. **`check_airspace_conflicts()`** — pure-geometry conflict detection (horizontal intersection + vertical altitude overlap)
4. **`fetch_and_check()`** — one-call convenience that fetches nearby airspaces and checks for conflicts

This notebook demonstrates the full workflow using airspace around
Santa Barbara / Vandenberg Space Force Base, California.

In [ ]:
import os

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from dotenv import load_dotenv

from hyplan.airspace import (
    Airspace, AirspaceConflict, OpenAIPClient,
    check_airspace_conflicts, fetch_and_check, clear_airspace_cache,
)
from hyplan.flight_line import FlightLine
from hyplan.units import ureg

# Load API keys from .env file in the project root
load_dotenv(os.path.join(os.path.dirname(os.getcwd()), ".env"))
load_dotenv()

## 1. Fetch Airspace Data from OpenAIP

The `OpenAIPClient` queries the OpenAIP API for airspace within a bounding box.
Results are cached locally for 24 hours (configurable via `max_age_hours`).
Use `clear_airspace_cache()` to force a fresh fetch.

In [ ]:
api_key = os.environ.get("OPENAIP_API_KEY", "")
assert api_key, "Set OPENAIP_API_KEY in your environment or .env file"

# Clear stale cache (optional — comment out to use cached data)
clear_airspace_cache()

# Fetch airspace for the Santa Barbara / Vandenberg area
client = OpenAIPClient(api_key=api_key)
airspaces = client.fetch_airspaces(
    bounds=(-121.0, 34.2, -119.5, 35.0),
    country="US",
)

print(f"Fetched {len(airspaces)} airspace zones:\n")
for a in airspaces:
    print(f"  {a.name:50s}  {a.airspace_class:12s}  "
          f"{a.floor_ft:>8,.0f} – {a.ceiling_ft:>8,.0f} ft")

## 2. Create Flight Lines

We'll create several flight lines at different locations and altitudes:

- **Line A** — flies through the Vandenberg restricted zone at 2,500 ft (conflict expected)
- **Line B** — flies through the danger zone at 10,000 ft (conflict expected)
- **Line C** — flies east of Santa Barbara, clear of all zones (no conflict)
- **Line D** — high-altitude coastal transect at 20,000 ft (may conflict with R-2534B which goes to FL999)

In [ ]:
# Line A — through Vandenberg restricted zone at 2,500 ft
line_a = FlightLine.start_length_azimuth(
    lat1=34.75, lon1=-120.8, length=ureg.Quantity(50, "km"), az=60,
    altitude_msl=ureg.Quantity(2500, "foot"), site_name="Line A (low, Vandenberg)",
)

# Line B — through the danger zone at 10,000 ft
line_b = FlightLine.start_length_azimuth(
    lat1=34.55, lon1=-120.6, length=ureg.Quantity(30, "km"), az=45,
    altitude_msl=ureg.Quantity(10000, "foot"), site_name="Line B (danger zone)",
)

# Line C — east of Santa Barbara, clear of all zones
line_c = FlightLine.start_length_azimuth(
    lat1=34.5, lon1=-119.6, length=ureg.Quantity(40, "km"), az=0,
    altitude_msl=ureg.Quantity(8000, "foot"), site_name="Line C (clear)",
)

# Line D — high altitude through R-2534B area
line_d = FlightLine.start_length_azimuth(
    lat1=34.42, lon1=-120.55, length=ureg.Quantity(30, "km"), az=90,
    altitude_msl=ureg.Quantity(20000, "foot"), site_name="Line D (high, R-2534B)",
)

flight_lines = [line_a, line_b, line_c, line_d]
for fl in flight_lines:
    print(f"  {fl.site_name:35s}  alt = {fl.altitude_msl.to('foot'):,.0f}")

## 3. Run Conflict Detection

`check_airspace_conflicts()` tests each flight line against each airspace for
both horizontal (polygon intersection) and vertical (altitude overlap) conflicts.
It uses a Shapely STRtree spatial index for efficient lookups.

In [ ]:
conflicts = check_airspace_conflicts(flight_lines, airspaces)

print(f"Found {len(conflicts)} conflict(s):\n")
for c in conflicts:
    fl = flight_lines[c.flight_line_index]
    print(f"  Flight line: {fl.site_name}")
    print(f"  Airspace:    {c.airspace.name} ({c.airspace.airspace_class})")
    print(f"  Vertical:    {c.vertical_overlap_ft[0]:,.0f} – {c.vertical_overlap_ft[1]:,.0f} ft MSL")
    print()

## 4. Visualize Conflicts

Plot airspace zones, flight lines, and highlight the portions that conflict.
Conflicting segments are shown in red; clear lines in green.

In [ ]:
def plot_airspace_conflicts(airspaces, flight_lines, conflicts, title="Airspace Conflict Detection"):
    """Plot airspaces, flight lines, and conflicts on a cartopy basemap."""
    fig, ax = plt.subplots(figsize=(12, 9), subplot_kw={"projection": ccrs.PlateCarree()})

    # Basemap features
    ax.add_feature(cfeature.LAND, facecolor="#f0efe6")
    ax.add_feature(cfeature.OCEAN, facecolor="#d6eaf8")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linestyle=":", linewidth=0.5)
    ax.add_feature(cfeature.LAKES, facecolor="#d6eaf8", edgecolor="gray", linewidth=0.5)
    ax.add_feature(cfeature.STATES, linewidth=0.3, edgecolor="gray")

    # Color map for airspace classes
    as_colors = {
        "RESTRICTED": ("red", 0.25),
        "DANGER": ("orange", 0.25),
        "B": ("blue", 0.20),
        "C": ("purple", 0.20),
        "D": ("teal", 0.20),
        "E": ("steelblue", 0.15),
        "OTHER": ("gray", 0.15),
    }

    # Plot airspace polygons
    seen_classes = set()
    for a in airspaces:
        color, alpha = as_colors.get(a.airspace_class, ("gray", 0.15))
        seen_classes.add(a.airspace_class)
        polys = list(a.geometry.geoms) if a.geometry.geom_type == "MultiPolygon" else [a.geometry]
        for poly in polys:
            x, y = poly.exterior.xy
            ax.fill(x, y, alpha=alpha, color=color, transform=ccrs.PlateCarree())
            ax.plot(x, y, "--", color=color, linewidth=1, alpha=0.6,
                    transform=ccrs.PlateCarree())
        cx, cy = a.geometry.centroid.coords[0]
        ax.text(cx, cy, f"{a.name}\n{a.floor_ft:,.0f}–{a.ceiling_ft:,.0f} ft",
                ha="center", va="center", fontsize=6, color=color, weight="bold",
                transform=ccrs.PlateCarree())

    # Track which flight lines have conflicts
    conflicting_fl_indices = {c.flight_line_index for c in conflicts}

    # Plot flight lines
    for i, fl in enumerate(flight_lines):
        lons, lats = zip(*fl.geometry.coords)
        color = "red" if i in conflicting_fl_indices else "green"
        ax.plot(lons, lats, "-o", color=color, linewidth=2, markersize=5,
                transform=ccrs.PlateCarree())
        ax.annotate(fl.site_name, xy=(lons[0], lats[0]), fontsize=7,
                    xytext=(5, 5), textcoords="offset points")

    # Plot intersection geometries
    for c in conflicts:
        geom = c.horizontal_intersection
        parts = list(geom.geoms) if geom.geom_type.startswith("Multi") else [geom]
        for part in parts:
            if hasattr(part, "coords") and len(part.coords) >= 2:
                x, y = zip(*part.coords)
                ax.plot(x, y, color="red", linewidth=5, alpha=0.5,
                        transform=ccrs.PlateCarree())

    # Legend
    legend_items = [
        mpatches.Patch(color="red", alpha=0.3, label="Conflicting line / intersection"),
        mpatches.Patch(color="green", alpha=0.5, label="Clear line"),
    ]
    for cls in sorted(seen_classes):
        color, _ = as_colors.get(cls, ("gray", 0.15))
        legend_items.append(mpatches.Patch(color=color, alpha=0.3, label=f"Airspace ({cls})"))
    ax.legend(handles=legend_items, loc="lower right", fontsize=8)

    # Set extent with a small buffer around all features
    all_lons, all_lats = [], []
    for fl in flight_lines:
        lons, lats = zip(*fl.geometry.coords)
        all_lons.extend(lons)
        all_lats.extend(lats)
    for a in airspaces:
        polys = list(a.geometry.geoms) if a.geometry.geom_type == "MultiPolygon" else [a.geometry]
        for p in polys:
            x, y = p.exterior.xy
            all_lons.extend(x)
            all_lats.extend(y)
    buf = 0.15
    ax.set_extent([min(all_lons) - buf, max(all_lons) + buf,
                   min(all_lats) - buf, max(all_lats) + buf],
                  crs=ccrs.PlateCarree())

    ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.3)
    ax.set_title(title, fontsize=13)
    plt.tight_layout()
    plt.show()

plot_airspace_conflicts(
    airspaces, flight_lines, conflicts,
    title="Airspace Conflicts — Santa Barbara / Vandenberg",
)

## 5. Conflict Matrix

A conflict requires **both** horizontal intersection (the flight line crosses the
airspace polygon) **and** vertical overlap (the flight altitude falls within
the airspace's floor–ceiling range). This matrix shows the relationship for
every flight line / airspace pair.

In [ ]:
from shapely import STRtree

# Build lookup of confirmed conflicts
conflict_pairs = {(c.flight_line_index, id(c.airspace)) for c in conflicts}

# Check horizontal intersection for all pairs
as_geoms = [a.geometry for a in airspaces]
tree = STRtree(as_geoms)

# Classify each (flight_line, airspace) pair
# Categories: "conflict", "horiz_only" (intersects but above/below), "none"
matrix = []  # rows=flight lines, cols=airspaces
for fl_idx, fl in enumerate(flight_lines):
    row = []
    candidates = set(tree.query(fl.geometry, predicate="intersects"))
    for as_idx, a in enumerate(airspaces):
        if as_idx in candidates:
            if (fl_idx, id(a)) in conflict_pairs:
                row.append("conflict")
            else:
                row.append("horiz_only")
        else:
            row.append("none")
    matrix.append(row)

# Plot
fig, ax = plt.subplots(figsize=(14, 4))

color_map = {"conflict": "#d32f2f", "horiz_only": "#ff9800", "none": "#e0e0e0"}
label_map = {"conflict": "CONFLICT\n(horiz + vert)", "horiz_only": "Horiz only\n(alt clear)", "none": "No intersection"}

for fl_idx in range(len(flight_lines)):
    for as_idx in range(len(airspaces)):
        status = matrix[fl_idx][as_idx]
        color = color_map[status]
        rect = plt.Rectangle((as_idx, fl_idx), 1, 1, facecolor=color, edgecolor="white", linewidth=2)
        ax.add_patch(rect)
        # Add text annotation
        fl = flight_lines[fl_idx]
        a = airspaces[as_idx]
        alt_ft = fl.altitude_msl.to("foot").magnitude
        if status == "conflict":
            txt = f"{alt_ft:,.0f} ft\nin {a.floor_ft:,.0f}–{a.ceiling_ft:,.0f}"
            ax.text(as_idx + 0.5, fl_idx + 0.5, txt, ha="center", va="center",
                    fontsize=6, color="white", weight="bold")
        elif status == "horiz_only":
            txt = f"{alt_ft:,.0f} ft\nvs {a.floor_ft:,.0f}–{a.ceiling_ft:,.0f}"
            ax.text(as_idx + 0.5, fl_idx + 0.5, txt, ha="center", va="center",
                    fontsize=6, color="white")

ax.set_xlim(0, len(airspaces))
ax.set_ylim(0, len(flight_lines))
ax.set_xticks([i + 0.5 for i in range(len(airspaces))])
ax.set_xticklabels([a.name for a in airspaces], rotation=45, ha="right", fontsize=7)
ax.set_yticks([i + 0.5 for i in range(len(flight_lines))])
fl_labels = [f"{fl.site_name}\n({fl.altitude_msl.to('foot'):,.0f})" for fl in flight_lines]
ax.set_yticklabels(fl_labels, fontsize=8)
ax.invert_yaxis()

# Legend
legend_items = [
    mpatches.Patch(color=color_map["conflict"], label="CONFLICT (horizontal + vertical overlap)"),
    mpatches.Patch(color=color_map["horiz_only"], label="Horizontal intersection only (altitude clear)"),
    mpatches.Patch(color=color_map["none"], label="No intersection"),
]
ax.legend(handles=legend_items, loc="upper center", bbox_to_anchor=(0.5, -0.35),
          ncol=3, fontsize=8)

ax.set_title("Flight Line / Airspace Conflict Matrix", fontsize=12)
plt.tight_layout()
plt.show()

## 6. One-Call Convenience: `fetch_and_check()`

For the simplest workflow, `fetch_and_check()` computes a bounding box from
your flight lines, fetches nearby airspaces, and runs the conflict check — all
in one call.

In [ ]:
quick_conflicts = fetch_and_check(
    flight_lines,
    api_key=api_key,
    buffer_m=40000,
    country="US",
)

print(f"Found {len(quick_conflicts)} conflict(s):\n")
for c in quick_conflicts:
    fl = flight_lines[c.flight_line_index]
    print(f"  {fl.site_name}  ↔  {c.airspace.name} ({c.airspace.airspace_class})")

## Summary

| Function | Purpose |
|----------|---------|
| `Airspace(name, airspace_class, ..., floor_ft, ceiling_ft, geometry)` | Dataclass representing an airspace volume (altitudes in feet MSL) |
| `AirspaceConflict` | Dataclass with conflict details (airspace, line index, intersection geometry, vertical overlap) |
| `check_airspace_conflicts(flight_lines, airspaces)` | Pure-geometry conflict detection with STRtree spatial index |
| `OpenAIPClient(api_key=)` | Fetches real airspace data from OpenAIP with local caching |
| `fetch_and_check(flight_lines, api_key=, buffer_m=)` | One-call convenience: fetch + check |
| `clear_airspace_cache()` | Remove cached data to force a fresh API fetch |

Key points:
- Conflict requires **both** horizontal intersection **and** vertical altitude overlap
- A flight line above an airspace's ceiling or below its floor is **not** a conflict
- The STRtree spatial index makes checks efficient even with many airspaces
- Flight levels (e.g. FL999) are correctly converted to feet (99,900 ft)
- API results are cached locally for 24 hours by default — use `clear_airspace_cache()` to refresh